# ДЗ 06 · NN-06 / Homework 06 · NN-06

**100 raw points.** Implement four NumPy blocks (80) and validation-based selection in a controlled experiment (20). / Реализуйте четыре блока NumPy (80) и выбор по validation в контролируемом эксперименте (20).

Впишите имя и группу; сдайте выполненный .ipynb с выводами. Не меняйте тесты и данные. / Enter your name and group; submit the executed .ipynb with explanations. Do not modify tests or data.

Срок вашей группы: [матрица](https://docs.google.com/spreadsheets/d/15N5GGHujXPAzRNuyOfmooT7A7sKhXgrMfWHkUkvYYiM/edit). Штраф за срок считается снаружи по времени сдачи, затем при отсутствии практики лекции 06 итог ×0.9 один раз. / Group deadline is in the matrix. Lateness is applied externally using verified submission time, then missing lecture 06 practice multiplies the result by 0.9 once.

Raw JSON is provisional: trusted tests, identity, deadlines, attendance and written conclusions are reviewed externally. / JSON не является окончательной оценкой; проверяющий сверяет тесты, личность, сроки, практику и выводы.

**Данные / Data.** UCI Optical Recognition of Handwritten Digits (E. Alpaydin, C. Kaynak, 1998), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), [UCI](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits), [loader documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). UCI lists CC BY 4.0; retain attribution. `load_digits` bundles 1,797 real 8×8 observations copied from the original UCI test portion. One observation = one handwritten digit; 64 block-count features in 0…16; target = digit 0…9. / UCI указывает CC BY 4.0; сохраняйте атрибуцию. 1 797 реальных изображений, одно наблюдение — цифра; 64 признака 0…16, цель — класс 0…9.

**Протокол / Protocol.** Fixed stratified teaching split: 1,078 train / 359 validation / 360 test, seed 606. Divide by known range 16; no fitted preprocessing on test. This is NOT the original benchmark split. Writer IDs are unavailable: no writer-disjoint generalization claim. Never tune using test. / Фиксированное стратифицированное учебное разбиение; это НЕ официальный benchmark. Нет идентификаторов авторов: независимость почерков не гарантируется. Test не используется для выбора модели.

**Среда / Environment.** Google Colab CPU; NumPy, scikit-learn, matplotlib, PyTorch. Tested with NumPy 2.1.3 and torch 2.8.0; data bundled with sklearn, no dataset download. / Данные входят в sklearn, скачивание датасета не требуется.

In [ ]:
student_first = ""
student_last = ""
student_group = ""
assignment_id = "NN-06"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
import copy, json
np.set_printoptions(precision=5, suppress=True)
torch.set_num_threads(1)
torch.manual_seed(606)
print('NumPy', np.__version__, 'PyTorch', torch.__version__)
def data_split():
 d=load_digits();X=d.data.astype(np.float64)/16.;y=d.target
 tr,rest=train_test_split(np.arange(len(y)),test_size=.4,stratify=y,random_state=606)
 va,te=train_test_split(rest,test_size=.5,stratify=y[rest],random_state=606)
 assert not(set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
 return X,y,tr,va,te
X,y,tr,va,te=data_split()
print(X.shape, len(tr), len(va), len(te))
def init_params(width=32,seed=606):
 r=np.random.default_rng(seed)
 return dict(W1=r.normal(0,np.sqrt(2/64),(64,width)),b1=np.zeros(width),W2=r.normal(0,np.sqrt(2/width),(width,10)),b2=np.zeros(10))

## A · loss_and_grad(z,y) · 20

RU: Верните (средний cross-entropy, градиент по logits). В каждой строке 10 классов, y — целые метки. Устойчивость при больших logits, неизменность при сдвиге строки. Не используйте torch внутри решения.

EN: Return (mean cross-entropy, gradient with respect to logits). Ten classes per row, integer y. Handle large logits and row-wise shifts stably. Do not use torch inside your implementation.

In [ ]:
def loss_and_grad(z,y):
    raise NotImplementedError("Implement stable loss and gradient")

## B · forward(X,p) · 20

RU: Верните (Z, (X,A,H)), где A=XW1+b1, H=ReLU(A), Z=HW2+b2. Не изменяйте входы.

EN: Return (Z, (X,A,H)), with A=XW1+b1, H=ReLU(A), Z=HW2+b2. Do not mutate inputs.

In [ ]:
def forward(X,p):
    raise NotImplementedError("Implement forward")

## C · backward(g,cache,p) · 30

RU: g — уже производная среднего loss по logits. Верните словарь градиентов W1,b1,W2,b2; формы совпадают с p. Не делите на B второй раз, не меняйте p.

EN: g is already the derivative of mean loss with respect to logits. Return gradients W1,b1,W2,b2 matching p shapes. Do not divide by B twice or mutate p.

In [ ]:
def backward(g,cache,p):
    raise NotImplementedError("Implement backward")

## D · sgd_step(p,g,lr) · 10

RU: Верните новый словарь p−lr*g, исходные p и g не меняются.

EN: Return a new dictionary p−lr*g; keep inputs unchanged.

In [ ]:
def sgd_step(p,g,lr):
    raise NotImplementedError("Implement SGD")

## E · select_best(records) · 20

RU: Верните индекс записи с минимальным val_loss, при равенстве — первый. Не используйте train loss или test. Затем запустите эксперимент ниже. 20 баллов проверяют корректность выбора; график и содержательный вывод проверяет преподаватель.

EN: Return the index of the minimum val_loss, choosing the first on ties. Do not use training loss or test. Then run the experiment below. The 20 points check selection correctness; the instructor reviews the plot and substantive conclusion.

In [ ]:
def select_best(records):
    raise NotImplementedError("Select using validation loss only")

## Контролируемый эксперимент / Controlled experiment

Две ширины: 16 и 32, lr=0.1, 45 эпох, seed=606. Не подбирайте по test. / Two widths: 16 and 32, lr=0.1, 45 epochs, seed=606. Do not select using test.

In [ ]:
def train(width,lr,epochs=45):
 X,y,tr,va,te=data_split();p=init_params(width);rng=np.random.default_rng(606);hist=[];best=None;bestloss=float('inf')
 for epoch in range(epochs):
  for ids in np.array_split(rng.permutation(tr),17):
   z,cache=forward(X[ids],p);_,g=loss_and_grad(z,y[ids]);p=sgd_step(p,backward(g,cache,p),lr)
  tl=loss_and_grad(forward(X[tr],p)[0],y[tr])[0];vl=loss_and_grad(forward(X[va],p)[0],y[va])[0];hist.append([tl,vl])
  if vl<bestloss:bestloss=vl;best={k:v.copy() for k,v in p.items()};ep=epoch+1
 return best,dict(width=width,lr=lr,best_epoch=ep,val_loss=bestloss,history=hist)
experiment_result=None
try:
 # These candidates are fixed before test / Кандидаты заданы до test.
 candidates=[train(16,.1),train(32,.1)]
 reports=[r for p,r in candidates]
 winner=select_best(reports)
 chosen,report=candidates[winner]
 # One final test evaluation / Единственная финальная оценка test.
 test_prediction=forward(X[te],chosen)[0].argmax(1)
 experiment_result={'selected':int(winner),'validation_losses':[r['val_loss'] for r in reports],
                    'test_accuracy':float(np.mean(test_prediction==y[te])),
                    'split_sizes':[len(tr),len(va),len(te)]}
 print(json.dumps(experiment_result,indent=2))
 for i,r in enumerate(reports):plt.plot(np.array(r['history'])[:,1],label=f"width={r['width']}")
 plt.xlabel('epoch');plt.ylabel('validation loss');plt.legend();plt.show()
except NotImplementedError:
 print('Сначала реализуйте функции / Implement the functions first')

## Вывод / Discussion

RU: Укажите выбранную ширину и эпоху, сравните validation loss, объясните три ошибки на реальных test-изображениях и ограничение по авторам. Покажите эти изображения своим кодом. Не меняйте модель после анализа test.

EN: State the selected width and epoch, compare validation losses, and explain three errors on actual test images plus the writer limitation. Display those images using your own code. Do not modify the model after test analysis.

**Ваш ответ / Your answer:**

In [ ]:
# Визуализация ошибок / Error visualization

In [ ]:
# Trusted public checks / Публичные проверки: не изменять / do not edit.
checks=[]
def check(name,points,test):
 try:
  test();checks.append({'test':name,'points':points,'passed':True})
 except Exception as exc:
  checks.append({'test':name,'points':0,'passed':False,'error':type(exc).__name__+': '+str(exc)[:160]})

def oracle(xb,yb,p):
 tp={k:torch.tensor(v,dtype=torch.float64,requires_grad=True) for k,v in p.items()}
 z=torch.relu(torch.tensor(xb,dtype=torch.float64)@tp['W1']+tp['b1'])@tp['W2']+tp['b2']
 loss=torch.nn.functional.cross_entropy(z,torch.tensor(yb,dtype=torch.long));loss.backward()
 return z.detach().numpy(),float(loss.detach()),{k:v.grad.numpy() for k,v in tp.items()}

def test_loss():
 for ids in [tr[:1],tr[10:21],tr[30:62]]:
  p=init_params(16);z,_,_=oracle(X[ids],y[ids],p)
  for logits in [z,z+10000,z*100]:
   t=torch.tensor(logits,dtype=torch.float64,requires_grad=True);ref=torch.nn.functional.cross_entropy(t,torch.tensor(y[ids]));ref.backward()
   l,g=loss_and_grad(logits,y[ids]);assert np.isfinite(l) and np.all(np.isfinite(g))
   np.testing.assert_allclose(l,float(ref.detach()),rtol=1e-8,atol=1e-9)
   np.testing.assert_allclose(g,t.grad.numpy(),rtol=1e-7,atol=1e-9)

def test_forward():
 for width in [16,32]:
  p=init_params(width);old={k:v.copy() for k,v in p.items()};xb=X[tr[:9]].copy()
  z,cache=forward(xb,p);ref,_,_=oracle(xb,y[tr[:9]],p)
  np.testing.assert_allclose(z,ref,atol=1e-10);assert len(cache)==3
  np.testing.assert_allclose(cache[0],xb);np.testing.assert_allclose(cache[1],xb@p['W1']+p['b1']);np.testing.assert_allclose(cache[2],np.maximum(cache[1],0))
  for k in p:np.testing.assert_array_equal(p[k],old[k])

def test_backward():
 for ids in [tr[:1],tr[:13],tr[40:64]]:
  p=init_params(16);old={k:v.copy() for k,v in p.items()};xb=X[ids]
  a=xb@p['W1']+p['b1'];h=np.maximum(a,0);z=h@p['W2']+p['b2']
  t=torch.tensor(z,dtype=torch.float64,requires_grad=True);torch.nn.functional.cross_entropy(t,torch.tensor(y[ids])).backward()
  got=backward(t.grad.numpy(),(xb,a,h),p);_,_,ref=oracle(xb,y[ids],p)
  assert set(got)==set(p)
  for k in p:
   assert got[k].shape==p[k].shape;np.testing.assert_allclose(got[k],ref[k],rtol=1e-7,atol=1e-9);np.testing.assert_array_equal(p[k],old[k])

def test_sgd():
 p=init_params(16);_,_,g=oracle(X[tr[:7]],y[tr[:7]],p);old={k:v.copy() for k,v in p.items()};oldg={k:v.copy() for k,v in g.items()}
 for lr in [0,.01,.2]:
  new=sgd_step(p,g,lr);assert set(new)==set(p)
  for k in p:
   np.testing.assert_allclose(new[k],old[k]-lr*g[k]);np.testing.assert_array_equal(p[k],old[k]);np.testing.assert_array_equal(g[k],oldg[k])

def test_selection():
 # Losses from real validation images; order and duplicate a record to test ties.
 records=[]
 for width in [8,16,32]:
  _,vl,_=oracle(X[va[:20]],y[va[:20]],init_params(width));records.append({'width':width,'val_loss':vl})
 for rr in [records,records[::-1],records+[records[0]], [records[0],records[0]]]:
  assert select_best(rr)==min(range(len(rr)),key=lambda i:rr[i]['val_loss'])
 assert experiment_result is not None, 'Run the controlled experiment'
 assert experiment_result['selected']==int(np.argmin(experiment_result['validation_losses']))
 assert experiment_result['split_sizes']==[1078,359,360]
 assert np.isfinite(experiment_result['test_accuracy']) and 0<=experiment_result['test_accuracy']<=1

check('A stable loss',20,test_loss)
check('B forward',20,test_forward)
check('C backward',30,test_backward)
check('D SGD',10,test_sgd)
check('E validation selection and experiment',20,test_selection)
raw_score=sum(r['points'] for r in checks)
print(json.dumps(checks,ensure_ascii=False,indent=2))
# No deadline or attendance penalties inside notebook.
payload={'student_first':student_first.strip(),'student_last':student_last.strip(),'student_group':student_group.strip(),
         'assignment_id':'NN-06','name':(student_first+' '+student_last).strip(),'group':student_group.strip(),
         'assignment':'NN-06','score':raw_score,'raw_score':raw_score,'max_score':100,'score_kind':'raw'}
print(json.dumps(payload,ensure_ascii=False))